In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Coding your own RNN

Using this pre-filled notebook, we will code our own RNN for sentence classification. For now, we'll keep using IMDB, as the goal of this part is to understand how an RNN works.

Unlike our previous lab, we will also learn the embedding layer. Which means we need to deal with vocabulary by ourselves.

In [3]:
from functools import partial
from typing import Callable, Dict, Generator, List, Tuple

from datasets import load_dataset
import numpy as np
from sklearn.utils import shuffle
import torch
from torch import nn

import torchtext
from torchtext.vocab import vocab, Vocab
from torchtext.data.utils import get_tokenizer

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from utils.rnn import (
    load_imdb_datasets,
    build_vocabulary,
    vectorize_text,
    data_generator,
    get_device,
    evaluate,
    train
)

The different functions from utils.rnn are in the file [rnn.py](./utils/rnn.py).

## Dataset
We load the dataset and split the training set in a stratified train/validation set.

In [4]:
train_df, valid_df, test_df = load_imdb_datasets()

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(valid_df)}")
print(f"Test set size: {len(test_df)}")

Train set size: 20000
Validation set size: 5000
Test set size: 25000


## Vocabulary (1 point)

**\[1 point\]** Build your own vocabulary. The [example provided in torchtext documentation](https://pytorch.org/text/stable/vocab.html#id1) might be of help.
* Don't forge to setup the `min_freq` parameter to not include unfrequent noise.
* You will need a tokenizer. Reuse the `basic_english` one from the our previous lab.
* For an RNN we need two special tokens: `<unk>`, for unknown words, and `<pad>` for padding.

In [5]:
tokenizer = get_tokenizer("basic_english")
vocabulary = build_vocabulary(dataset=train_df, tokenizer=tokenizer)

## Vectorize and batch the input (3 points)

As seen in class, our model should take one-hot encoded vectors corresponding to the each token vocabulary id. However, computing a vector x matrix multiplication for every input is unnecessarily costly. Multiplying a one-hot vector with a matrix is the equivalent of taking one row of the matrix. In pyTorch, we provide ids for each token which will be used as input to an `nn.Embedding` layer. The id is simply the row in the embedding matrix.

**\[1 point\]** Fill the `vectorize_text` function returning a 1D torch tensor of `torch.long` for each input text.

In [6]:
text_pipeline = partial(vectorize_text, vocabulary=vocabulary, tokenizer=tokenizer)

Check the function is working correctly, especially it should return the right special id for unknown words.

In [7]:
text_pipeline("Some text I am thinking about... ragafqfa")

tensor([  40, 5413,   11,  272,  664,    8,   27,   27,   27,    0])

In [8]:
X_train = [text_pipeline(text) for text in tqdm(train_df["text"])]
y_train = train_df["label"]
X_valid = [text_pipeline(text) for text in tqdm(valid_df["text"])]
y_valid = valid_df["label"]
X_test = [text_pipeline(text) for text in tqdm(test_df["text"])]
y_test = test_df["label"]

  0%|          | 0/20000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/25000 [00:00<?, ?it/s]

In [9]:
sample_rate = 1
X_train_sampled = X_train[::sample_rate]
y_train_sampled = y_train[::sample_rate]
X_valid_sampled = X_valid[::sample_rate]
y_valid_sampled = y_valid[::sample_rate]
X_test_sampled = X_test[::sample_rate]
y_test_sampled = y_test[::sample_rate]

To speed up the training process, we turn the inputs into batches, as we did last time. For batches to work, every line must have the same lengths. Last time, it was implicit as only a vector (the average of all embeddings) was provided. This time, every line has the length of a different review.

To go around this problem, we use padding. So every line within a batch is padded to the length of its longest element.

* **\[1 point\]** Fill the data generator function.
* **\[1 point\]** On which side should you pad and why?

In [ ]:
# pad_token = "<pad>"
# BATCH_SIZE = 32
# train_gen = lambda: data_generator(X_train, y_train, vocabulary[pad_token], batch_size=BATCH_SIZE)
# valid_gen = lambda: data_generator(X_valid, y_valid, vocabulary[pad_token], batch_size=BATCH_SIZE)
# test_gen = lambda: data_generator(X_test, y_test, vocabulary[pad_token], batch_size=BATCH_SIZE)

In [10]:
def data_generator_test(
    X: List[torch.tensor], y: List[int], pad_id: int, batch_size: int = 32
) -> Generator[Tuple[torch.Tensor, torch.Tensor], None, None]:
    """
    Yield batches from given input data and labels.
    Args:
        X: a list of tensor (input features).
        y: the corresponding labels.
        batch_size: the size of every batch [32].
    Returns:
        A tuple of tensors (features, labels).
    """
    X, y = shuffle(X, y)
    for i in range(0, len(X), batch_size):
        batch_X = X[i : i + batch_size]
        batch_y = y[i : i + batch_size]
        batch_X = torch.nn.utils.rnn.pad_sequence(
            [elt.flip(dims=[0]) for elt in batch_X],
            batch_first=True,
            padding_value=vocabulary[pad_token],
        ).flip(dims=[1])
        yield batch_X, torch.tensor(batch_y, dtype=torch.float32)

pad_token = "<pad>"
BATCH_SIZE = 128
train_gen = lambda: data_generator_test(X_train_sampled, y_train_sampled, vocabulary[pad_token], batch_size=BATCH_SIZE)
valid_gen = lambda: data_generator_test(X_valid_sampled, y_valid_sampled, vocabulary[pad_token], batch_size=BATCH_SIZE)
test_gen = lambda: data_generator_test(X_test_sampled, y_test_sampled, vocabulary[pad_token], batch_size=BATCH_SIZE)

In [11]:
device = get_device()

## Classifier (3 points)

**\[3 points\]** Code your own RNN. Fill the `RNN` class correctly. Remember an RNN has 3 matrices and an embedding layer (see course slide 61).
* The embedding layer turns a one-hot vectors into dense vectors.
* The first matrix (W) connects the embedding to the hidden layer.
  * `embedding_size -> hidden_size`
* The second matrix (U) connect the previous hidden layer to the current one.
  * `hidden_size -> hidden_size`
* These to vectors are added and go through an activation function (e.g. $h_t = tanh(Wx_i+Uh_{t-1})$).
* The last matrix (V) connects the hidden layer to the hidden layer to the output.
  * `hidden_size -> 1`
* Donc forget to add an `init_hidden` function which initialize the first hidden layer to 0.

In [12]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.W = nn.Linear(embed_size, hidden_size)
        self.U = nn.Linear(hidden_size, hidden_size)
        self.V = nn.Linear(hidden_size, output_size)
        self.tanh = nn.Tanh()

        self._init_weights()

    def _init_weights(self):
        # Input-to-hidden
        torch.nn.init.xavier_uniform_(self.W.weight)
        torch.nn.init.zeros_(self.W.bias)
        
        # Hidden-to-hidden (orthogonal)
        torch.nn.init.orthogonal_(self.U.weight)
        torch.nn.init.zeros_(self.U.bias)

        # Hidden-to-output
        torch.nn.init.xavier_uniform_(self.V.weight)
        torch.nn.init.zeros_(self.V.bias)

        # Embeddings
        torch.nn.init.uniform_(self.embedding.weight, -0.1, 0.1)

    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device=device)

    def forward(self, x):
        x = self.embedding(x)
        h = self.init_hidden(x.shape[0]).to(x.device)
        for i in range(x.shape[1]):
            h = self.tanh(self.W(x[:, i]) + self.U(h))
        return torch.sigmoid(self.V(h))

## Training (2 points)

Training is a bit different than usual. We will need to sequentially (but in "batch parallel") go through an input, keeping track of the hidden layer, and use the last output as prediction.

**\[2 point\]** Code the training loop.
* Note that for each batch, you need to loop through the whole input and use the output of the last token as input to your criterion.
* Keep the best model evaluated on the validation set.
* Plot the training and validation losses.
* Training will take some time (~30 min on a T4 GPU). Make sure your results appear in the notebook.

In [13]:
n_embedding = 32
n_hidden = 64
model = RNN(len(vocabulary.get_itos()), n_embedding, n_hidden, 1).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)

# For the training, we need to sequentially (but in "batch parallel") go through an input, keeping track of the hidden layer, and use the last output as prediction.
# For each batch, we need to loop through the whole input and use the output of the last token as input for our criterion.
# Keep the best model evaluated on the validation set.adam
train_losses, valid_losses = train(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_generator=train_gen,
    valid_generator=valid_gen,
    n_epochs=5,
    device=device
)

# Plot the train and validation losses.
plt.plot(train_losses, label="train loss")
plt.plot(valid_losses, label="valid loss")
plt.legend()
plt.show()

/home/fsoulier/learning/NLP-2023/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:218: UserWarning: 
NVIDIA RTX PRO 4000 Blackwell Generation Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 4000 Blackwell Generation Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/157 [00:00<?, ?it/s]

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Evaluation (1 point)

* **\[1 point\]** Compute the accuracy for all 3 splits.

In [ ]:
# Evaluate the model on the train set.
accuracy, precision, recall, f1 = evaluate(model, train_gen, device)

In [ ]:
# Evaluate the model on the validation set.
_ = evaluate(model, valid_gen, device)

In [ ]:
# Evaluate the model on the test set.
_ = evaluate(model, test_gen, device)

Strangely enough, we have an isssue with the accuracy. As we can observe during the training, the loss remains approximately constant. Here, the testing results are also identical for all the datasets. Unfortunately, we could not find the reason for this issue ...